# Imports

In [9]:
import sys
from pathlib import Path
project_root = Path().resolve().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.utils import *
from src.hmm import HMM
from src.analysis import *
from src.viterbi import viterbi

import math
import random
from scipy.stats import ttest_ind
from pprint import pprint

# Train HMM

In [2]:
states = ["adapted", "not_adapted"]
filename = "../data/GCF_000001405.40_GRCh38.p14_cds_from_genomic.fna.gz"
hmm = HMM(states)
hmm.initialize_parameters()
hmm.train_emission_probs_from_fasta(fasta_filename=filename)

# Synthetic Data

In [10]:
def generate_sample_sequence(codons, probs, n):
    return "".join(random.choices(codons, weights=probs, k=n))
    

codon_list = generate_all_codons()
probs = [math.exp(hmm.emission_probs["adapted"][codon]) for codon in codon_list]

human_like_seq = {}
random_seq = {}

for i in range(1, 21):
    human_like_seq[f"h{i}"] = generate_sample_sequence(codon_list, probs, 20)
    random_seq[f"r{i}"] = generate_sample_sequence(codon_list, [1/64]*64, 20)

print("Human-like sequences:")
pprint(human_like_seq)
print("Random sequences:")
pprint(random_seq)

Human-like sequences:
{'h1': 'ATGGCATGTTCCACTATCAGGGCCACCGAGGAGACGAGACAGGTTCTCTTGTGTCAGTTT',
 'h10': 'CAAAGCTTCGCACGCGACGTCAAATCTCTAGAGAACGGTGCTACCTGGCTTATGAAAGTG',
 'h11': 'GGAATCTCCTCACAGCAGCCTGAGAGGAGCTTCTTAGGATCCAAACAAGAAATCCACTGG',
 'h12': 'ACCCATCAGATGCCCTCCAAATTTTCCGCCCTCAGCCCCGTGACCGAGCAGTTAACTGCC',
 'h13': 'CTGGGGGTGTCCATCCTGCTGGTGCTGAAAGGTGCGGAGGAGCAGAGTCTTATGTCTGAC',
 'h14': 'CGAACAAACGTGGGCATCGGAGGCGGCCGCACTAAAATTATGGCAGGCGCGGAGTACAAG',
 'h15': 'CTTGAAAGCGTGCTATATACTATCTCTGTGAACCCCAAGCGGGTTCCTAGAGAGTTGACC',
 'h16': 'GAGCCCGCCCTCCATCTGGATCATAAGAATGCTGGCAGGAACACCTACGTGATTATGTTT',
 'h17': 'GGCCTGATGGAGCCTGGGGAACTACATTCCTTCTTCAGGGTACAGCAGAGGATTGGTATT',
 'h18': 'CGGGAGACACCTCTGTTCCAGGGGATCTCCCGTTCAGTGAACTATCTCCTGGTAACCACG',
 'h19': 'GGGCTGGTGCGGGCACTGGAAGTAGCAAGATTCGAAACCGAGACACTGAAAAATACTGAC',
 'h2': 'TGCAACGCCTCTTCAGATGAAGATCTTGCTAGCCAACTGCTGGAGCCCAATGGGCTTAGT',
 'h20': 'CCCGGACCTATAGTGCTGGAGCTCCGTGTAGTGTATCTTGCGGTCAGAGCCAGACCACAT',
 'h3': 'CTGGGGTCCACAAGTGAGTCCAAGGATGATAAGTAC

# Test with synthetic data

In [11]:
results_human = analyze_genes(human_like_seq, hmm)
results_random = analyze_genes(random_seq, hmm)

human_scores = []
random_scores = []
for score, _ in results_human.values():
    human_scores.append(score)

for score, _ in results_random.values():
    random_scores.append(score)

human_avg = sum(human_scores)/len(human_scores)
random_avg = sum(random_scores)/len(random_scores)

print("Human sequences:")
pprint(results_human)
print("Random sequences:")
pprint(results_random)

print(f"Human-like average: {human_avg}")
print(f"Random average: {random_avg}")

Human sequences:
{'h1': (1.0,
        ['adapted',
         'adapted',
         'adapted',
         'adapted',
         'adapted',
         'adapted',
         'adapted',
         'adapted',
         'adapted',
         'adapted',
         'adapted',
         'adapted',
         'adapted',
         'adapted',
         'adapted',
         'adapted',
         'adapted',
         'adapted',
         'adapted',
         'adapted']),
 'h10': (1.0,
         ['adapted',
          'adapted',
          'adapted',
          'adapted',
          'adapted',
          'adapted',
          'adapted',
          'adapted',
          'adapted',
          'adapted',
          'adapted',
          'adapted',
          'adapted',
          'adapted',
          'adapted',
          'adapted',
          'adapted',
          'adapted',
          'adapted',
          'adapted']),
 'h11': (1.0,
         ['adapted',
          'adapted',
          'adapted',
          'adapted',
          'adapted',
          'ad

# Statistical test

In [12]:
t_stat, p_val = ttest_ind(human_scores, random_scores)

print("t-stat:", t_stat)
print("p-value:", p_val)

t-stat: 8.941001438966193
p-value: 6.963644174550132e-11
